<a href="https://colab.research.google.com/github/totaswift15/Arabic-new-classifier-related-to-Qatar-/blob/main/ver1_arabert_starter_notebook_Qatar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ZINDI Arabic Multi-Label Classification

## Setup

In [1]:
%pip install -q -U "huggingface-hub>=1.23.0,<2.0" "transformers>=4.48.0" datasets accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.9/842.9 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 88.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 36.7 MB/s eta 0:00:00


In [2]:
%pip install -q -U google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 26.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.0 which is incompatible.


## Imports

In [3]:
# Standard library imports
import hashlib
import json
import os
import random
import re
import sqlite3
import threading
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

# Third-party library imports
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from pydantic import BaseModel, Field
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer, Trainer, TrainingArguments

# Google / Gemini specific imports
import google.generativeai as genai
from google import genai as genai_new
from google.genai import types
from google.colab import userdata

/usr/local/lib/python3.13/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


## Env Variables

In [4]:
import os
import re
import random
from pathlib import Path
import numpy as np
import torch

# 1. Reproducibility & Seed Setup
SEED = 42

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

# 2. Hardware Device Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 3. Directories & File Paths
DATA_DIR = Path("/content")
OUTPUT_DIR = Path("/content/submissions")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 4. Column Names & Target Definitions
ID_COLUMN = "ID"
TEXT_COLUMN = "text"
QATAR_COLUMN = "Qatar Related"
REMAINING_TARGETS = [
    "Geography",
    "Politics & Conflict",
    "Health & Wellbeing",
    "Science",
    "Sports",
]
TARGET_COLUMNS = [QATAR_COLUMN] + REMAINING_TARGETS

# 5. Cleaning Regex Expressions
ARABIC_DIACRITICS = re.compile(r"[\u0617-\u061a\u064b-\u0652]")
NON_TEXT = re.compile(r"[^\w\s\u0600-\u06ff]")
WHITESPACE = re.compile(r"\s+")

# 6. Model Training Hyperparameters
MODEL_NAME = "UBC-NLP/MARBERTv2"  # Primary MARBERT backbone
MAX_LEN = 512                      # Token sequence length
BATCH_SIZE = 16                    # Lower to 8 if GPU hits memory limits
EPOCHS = 4
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

print("Environment variables and hyperparameters initialized successfully!")

Using device: cuda
Environment variables and hyperparameters initialized successfully!


## Utils

In [5]:
import pandas as pd
from collections import Counter

# 1. Function to Load Data with Validation Checks
def load_data(data_dir=DATA_DIR):
    """
    Loads training and test data from CSV files and validates required columns.
    """
    train_frame = pd.read_csv(data_dir / "Train.csv")
    test_frame = pd.read_csv(data_dir / "Test.csv")

    # Define required columns for validation
    required_train = {ID_COLUMN, TEXT_COLUMN, *TARGET_COLUMNS}
    required_test = {ID_COLUMN, TEXT_COLUMN}

    # Check for missing columns
    missing_train = required_train.difference(train_frame.columns)
    missing_test = required_test.difference(test_frame.columns)

    if missing_train or missing_test:
        raise ValueError(
            f"Missing train columns: {missing_train}; missing test columns: {missing_test}"
        )
    return train_frame, test_frame

# 2. Function for Professional Arabic Text Cleaning
def clean_arabic_text(text: str) -> str:
    """
    Normalizes Arabic orthography, removes Tashkeel and Tatweel, and strips HTML/punctuation.
    """
    if not isinstance(text, str):
        return ""
    # Remove URLs and HTML tags
    text = re.sub(r"http\S+|www\.\S+|<.*?>", " ", text)
    # Remove Tashkeel (diacritics)
    text = re.sub(ARABIC_DIACRITICS, "", text)
    # Remove Tatweel (elongation ـ)
    text = re.sub(r"\u0640", "", text)
    # Normalize Alef variants (أ, إ, آ -> ا)
    text = re.sub(r"[\u0622\u0623\u0625]", "\u0627", text)
    # Normalize Alef Maqsoora (ى -> ي) and Hamza variants
    text = re.sub(r"\u0649", "\u064A", text)
    text = re.sub(r"[\u0624\u0626]", "\u0621", text)
    # Remove non-Arabic punctuation & normalize spaces
    text = re.sub(NON_TEXT, " ", text)
    text = re.sub(WHITESPACE, " ", text).strip()
    return text

# 3. Function for Multilabel Stratified Oversampling
def perform_multilabel_oversampling(df: pd.DataFrame, target_cols: list, min_samples_per_combo: int = 10) -> pd.DataFrame:
    """
    Identifies unique 6-label target combinations and oversamples rare joint classes.
    """
    df = df.copy()
    # Create a joint composite key across all 6 target dimensions
    df["combo_key"] = df[target_cols].astype(str).agg("_".join, axis=1)

    counts = df["combo_key"].value_counts()
    rare_combos = counts[counts < min_samples_per_combo].index

    oversampled_dfs = [df]
    for combo in rare_combos:
        subset = df[df["combo_key"] == combo]
        num_to_add = min_samples_per_combo - len(subset)
        if num_to_add > 0:
            resampled_subset = subset.sample(n=num_to_add, replace=True, random_state=SEED)
            oversampled_dfs.append(resampled_subset)

    balanced_df = pd.concat(oversampled_dfs, ignore_index=True).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    balanced_df.drop(columns=["combo_key"], inplace=True)
    return balanced_df

# --- Execution ---
print("1. Loading raw training and test data...")
train_df, test_df = load_data()

print("2. Cleaning Arabic text on Train dataset ONLY...")
train_df["cleaned_text"] = train_df[TEXT_COLUMN].apply(clean_arabic_text)

# Keep raw text in test_df for tokenization
test_df["cleaned_text"] = test_df[TEXT_COLUMN]

print("3. Applying multilabel oversampling on rare category combinations...")
original_len = len(train_df)
train_df = perform_multilabel_oversampling(train_df, TARGET_COLUMNS, min_samples_per_combo=10)

print(f"\nData Pipeline Complete!")
print(f"Original Train rows: {original_len} | Oversampled Train rows: {len(train_df)}")
print(f"Test rows (uncleaned raw text): {len(test_df)}")

1. Loading raw training and test data...
2. Cleaning Arabic text on Train dataset ONLY...
3. Applying multilabel oversampling on rare category combinations...

Data Pipeline Complete!
Original Train rows: 3786 | Oversampled Train rows: 5207
Test rows (uncleaned raw text): 1262


In [6]:
def preprocess_arabic(text):
    """
    Applies professional normalization steps to Arabic text for Transformer tokenization.

    Removes URLs, HTML tags, diacritics, Tatweel (kashida), normalizes Alef and
    Hamza variants, removes non-Arabic punctuation, and standardizes whitespace.

    Args:
        text (str): The Arabic text to preprocess.

    Returns:
        str: The preprocessed Arabic text.
    """
    if not isinstance(text, str):
        return ""

    # 1. Remove URLs and HTML tags
    text = re.sub(r"http\S+|www\.\S+|<.*?>", " ", text)

    # 2. Remove diacritics (Tashkeel)
    text = ARABIC_DIACRITICS.sub("", text)

    # 3. Remove Tatweel (elongation ـ)
    text = re.sub(r"\u0640", "", text)

    # 4. Normalize Alef variants (أ, إ, آ -> ا)
    text = re.sub(r"[\u0622\u0623\u0625]", "\u0627", text)

    # 5. Normalize Alef Maqsoora (ى -> ي) and Hamza variants
    text = re.sub(r"\u0649", "\u064A", text)
    text = re.sub(r"[\u0624\u0626]", "\u0621", text)

    # 6. Remove non-text characters and normalize whitespace
    return WHITESPACE.sub(" ", NON_TEXT.sub(" ", text)).strip()

In [7]:
import numpy as np
from sklearn.metrics import f1_score

def score_predictions(actual, predicted, target_columns=TARGET_COLUMNS):
    """
    Calculates the weighted F1-score for each target column and returns the mean F1-score.

    Args:
        actual (pd.DataFrame): DataFrame containing the true labels.
        predicted (pd.DataFrame): DataFrame containing the predicted labels.
        target_columns (list): A list of column names for which to calculate F1-scores.

    Returns:
        dict: A dictionary containing per-column weighted F1 scores and the overall 'mean'.
    """
    scores = {}
    for column in target_columns:
        # Convert to string to ensure safe comparison between integer/categorical predictions
        y_true = actual[column].astype(str)
        y_pred = predicted[column].astype(str)
        scores[column] = f1_score(y_true, y_pred, average="weighted")

    scores["mean"] = float(np.mean(list(scores.values())))
    return scores

In [8]:
from pathlib import Path
import pandas as pd

def make_submission(test_frame, predictions, output_path):
    """
    Formats predictions into the competition submission format and saves to CSV.

    Args:
        test_frame (pd.DataFrame): The original test dataframe containing ID_COLUMN.
        predictions (pd.DataFrame): DataFrame containing predicted labels for TARGET_COLUMNS.
        output_path (str or Path): The file path to save the submission CSV.

    Returns:
        pd.DataFrame: The formatted submission DataFrame.
    """
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    # 1. Create base submission dataframe aligned by ID
    submission = test_frame[[ID_COLUMN]].copy()

    # 2. Assign target column predictions in exact required order
    for col in TARGET_COLUMNS:
        submission[col] = predictions[col].values

    # 3. Validation Safety Checks
    if len(submission) != len(test_frame):
        raise ValueError(f"Row count mismatch! Test: {len(test_frame)}, Submission: {len(submission)}")

    if submission[TARGET_COLUMNS].isna().sum().sum() > 0:
        raise ValueError("Submission contains NaN/missing values! Check model inference.")

    # 4. Save to CSV
    submission.to_csv(output_path, index=False)
    print(f"Submission successfully saved to: {output_path}")
    print(f"Submission Shape: {submission.shape}")

    return submission

In [9]:
import torch
from torch.utils.data import Dataset
from transformers import AutoModel, AutoTokenizer

# Initialize tokenizer from environment config
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class NewsDataset(Dataset):
    """
    Custom PyTorch Dataset for multi-head news classification.
    Tokenizes text and formats labels for all 6 target dimensions.
    """

    def __init__(self, frame, is_test=False):
        self.is_test = is_test

        # Fallback to TEXT_COLUMN if cleaned_text is missing
        text_data = frame["cleaned_text"] if "cleaned_text" in frame.columns else frame[TEXT_COLUMN]

        self.encodings = tokenizer(
            text_data.fillna("").tolist(),
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt",
        )

        if not self.is_test:
            # Prepare tensor labels for ALL 6 target dimensions
            self.labels = {
                column: torch.tensor(
                    label_encoders[column].transform(frame[column].astype(str)),
                    dtype=torch.long,
                )
                for column in TARGET_COLUMNS
            }

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, index):
        item = {
            "input_ids": self.encodings["input_ids"][index],
            "attention_mask": self.encodings["attention_mask"][index],
        }
        if not self.is_test:
            item["labels"] = {
                column: values[index] for column, values in self.labels.items()
            }
        return item


class MultiHeadAraBERT(torch.nn.Module):
    """
    Multi-head Transformer model fine-tuning 6 independent classification heads
    (Qatar Related + 5 domain targets) on top of the MARBERT backbone.
    """

    def __init__(self, class_counts, dropout_prob=0.3):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(MODEL_NAME)
        hidden_size = self.encoder.config.hidden_size
        self.dropout = torch.nn.Dropout(dropout_prob)

        # Classification heads for ALL target columns
        self.heads = torch.nn.ModuleDict(
            {
                column: torch.nn.Linear(hidden_size, count)
                for column, count in class_counts.items()
            }
        )

    def forward(self, input_ids, attention_mask, labels=None):
        # Extract [CLS] token vector representation
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.dropout(outputs.last_hidden_state[:, 0])

        # Generate logits for each of the 6 target heads
        logits = {column: head(pooled) for column, head in self.heads.items()}

        if labels is None:
            return logits

        # Calculate Cross-Entropy Loss across all 6 target dimensions
        losses = [
            torch.nn.functional.cross_entropy(logits[column], labels[column])
            for column in TARGET_COLUMNS
        ]
        total_loss = torch.stack(losses).mean()
        return (total_loss, logits)

config.json:   0%|          | 0.00/757 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/439 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.10M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [10]:
import torch

def custom_data_collator(features):
    """
    A custom data collator for multi-head classification.
    Batch-stacks input token IDs, attention masks, and multi-head label dictionaries.

    Args:
        features (list): List of sample dictionaries from NewsDataset.

    Returns:
        dict: Batch dictionary containing stacked 'input_ids', 'attention_mask',
              and 'labels' dict (if training/validation).
    """
    # 1. Stack input features across the batch
    batch = {
        "input_ids": torch.stack([f["input_ids"] for f in features]),
        "attention_mask": torch.stack([f["attention_mask"] for f in features]),
    }

    # 2. Handle multi-head target labels if present
    if "labels" in features[0]:
        first_label = features[0]["labels"]

        if isinstance(first_label, dict):
            # Efficiently stack tensor labels per target head
            batch["labels"] = {
                key: torch.stack([f["labels"][key] for f in features])
                for key in first_label.keys()
            }
        else:
            # Fallback for standard single-label target tensors
            batch["labels"] = torch.stack([f["labels"] for f in features])

    return batch

In [11]:
import pandas as pd
import torch
from torch.utils.data import DataLoader

def predict_arabert(model, frame, is_test=False):
    """
    Generates predictions for the MARBERT multi-head model across all 6 target dimensions.

    Args:
        model (torch.nn.Module): The trained MultiHeadAraBERT model.
        frame (pd.DataFrame): The DataFrame containing text data to predict on.
        is_test (bool): Whether the frame is a test set.

    Returns:
        pd.DataFrame: A DataFrame containing inverse-transformed predicted labels
                      for all 6 target columns (including Qatar Related).
    """
    # 1. Create dataset and loader
    dataset = NewsDataset(frame, is_test=is_test)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

    # 2. Set model to evaluation mode
    model.to(device).eval()

    # 3. Initialize prediction containers for ALL 6 target columns
    predictions = {column: [] for column in TARGET_COLUMNS}

    # 4. Batched inference loop
    with torch.no_grad():
        for batch in loader:
            inputs = {
                "input_ids": batch["input_ids"].to(device),
                "attention_mask": batch["attention_mask"].to(device),
            }

            out = model(**inputs)
            logits = out[1] if isinstance(out, tuple) else out

            # Extract argmax class indices for all 6 heads
            for column in TARGET_COLUMNS:
                preds = logits[column].argmax(dim=1).cpu().tolist()
                predictions[column].extend(preds)

    # 5. Inverse-transform integer indices back to original string labels
    return pd.DataFrame({
        column: label_encoders[column].inverse_transform(predictions[column])
        for column in TARGET_COLUMNS
    })

## Preprocess text and create a validation split

This section applies a specific Arabic normalizer that handles diacritics, punctuation, and letter variations (such as normalizing 'Alif' forms, 'Ya', and 'Ta Marbuta'). After cleaning, the data is split into training and validation sets, using a stratified approach on the `Qatar Related` target to ensure balanced representation across both subsets.

In [12]:
from IPython.display import display

# 1. Load raw training and test data
raw_train_df, test_df = load_data()
print(f"Raw Train shape: {raw_train_df.shape} | Test shape: {test_df.shape}")

# 2. Display first 2 rows of training dataframe
print("\n--- Raw Training Data Sample ---")
display(raw_train_df.head(2))

# 3. Check for missing values in text and all 6 target columns
print("\n--- Missing Value Verification ---")
display(raw_train_df[[TEXT_COLUMN, *TARGET_COLUMNS]].isna().sum().to_frame("missing"))

# 4. Display class distributions for ALL 6 target columns
print("\n--- Target Class Distributions (Raw Data) ---")
for column in TARGET_COLUMNS:
    print(f"\n[Target Head: {column}]")
    display(raw_train_df[column].value_counts(dropna=False).to_frame("count"))

# 5. Prepare final train_df: Apply Arabic text cleaning (Train ONLY) and Multilabel Oversampling
print("\n--- Executing Data Preparation Pipeline ---")
train_df = raw_train_df.copy()
train_df["cleaned_text"] = train_df[TEXT_COLUMN].apply(clean_arabic_text)

# Preserve uncleaned raw text for test_df
test_df["cleaned_text"] = test_df[TEXT_COLUMN]

# Oversample rare multilabel target combinations
train_df = perform_multilabel_oversampling(train_df, TARGET_COLUMNS, min_samples_per_combo=10)

print(f"\nPipeline Complete!")
print(f"Original Train rows: {len(raw_train_df)} -> Processed & Oversampled Train rows: {len(train_df)}")
print(f"Uncleaned Test rows: {len(test_df)}")

Raw Train shape: (3786, 8) | Test shape: (1262, 2)

--- Raw Training Data Sample ---


,ID,text,Qatar Related,Geography,Politics & Conflict,Health & Wellbeing,Science,Sports
0,fL8IqbB5haFSy2gk,كشف مسؤول رياضي مصري أن الميدالية الذهبية التي...,1,G-7,NO_POLITICS,NO_HEALTH,NO_SCIENCE,SP-3
1,Jivkt5fbyLOBsaQd,أشاد وزير الدولة للشؤون الخارجية لدولة قطر، سل...,1,G-8,PC-1,NO_HEALTH,NO_SCIENCE,NO_SPORTS



--- Missing Value Verification ---


,missing
text,0
Qatar Related,0
Geography,0
Politics & Conflict,0
Health & Wellbeing,0
Science,0
Sports,0



--- Target Class Distributions (Raw Data) ---

[Target Head: Qatar Related]


...


[Target Head: Science]


,count
Science,
NO_SCIENCE,2605
SC-1,588
SC-3,321
SC-2,272



[Target Head: Sports]


,count
Sports,
NO_SPORTS,2314
SP-3,461
SP-4,370
SP-2,338
SP-1,303



--- Executing Data Preparation Pipeline ---

Pipeline Complete!
Original Train rows: 3786 -> Processed & Oversampled Train rows: 5207
Uncleaned Test rows: 1262


In [13]:
from sklearn.model_selection import train_test_split

# 1. Clean and normalize Arabic text for TRAIN set ONLY
train_df["processed_text"] = train_df[TEXT_COLUMN].fillna("").map(preprocess_arabic)

# 2. Keep TEST set raw (no text cleaning applied)
test_df["processed_text"] = test_df[TEXT_COLUMN].fillna("")

# 3. Split training set into 80% train / 20% validation
# Stratify on 'Qatar Related' to ensure equal binary class ratios in both splits
train_part, validation_part = train_test_split(
    train_df,
    test_size=0.2,
    random_state=SEED,
    stratify=train_df[QATAR_COLUMN],
)

print(f"Dataset Split Complete!")
print(f" - Training split: {len(train_part)} rows")
print(f" - Validation split: {len(validation_part)} rows")
print(f" - Test set (raw text preserved): {len(test_df)} rows")

Dataset Split Complete!
 - Training split: 4165 rows
 - Validation split: 1042 rows
 - Test set (raw text preserved): 1262 rows


## Gemini for `Qatar Related`

This intentionally basic prompt returns only `0` or `1`. Keep the API key in Colab Secrets or an environment variable.

### 1. Persistent Caching Mechanism
To save costs and avoid rate limits during experimentation, we use a SQLite-based disk cache. The cache key is derived from a hash of both the article text and the system prompt, ensuring that if you update your instructions, the model will re-run the classification.

In [14]:
import hashlib
import sqlite3
import threading
from pathlib import Path

# 1. Database file path configuration
CACHE_DB_PATH = Path("/content/gemini_cache.db")

# 2. Initialize SQLite Database & Table Schema
with sqlite3.connect(CACHE_DB_PATH) as conn:
    conn.execute(
        "CREATE TABLE IF NOT EXISTS predictions (key TEXT PRIMARY KEY, val TEXT)"
    )

# 3. Thread-local connection pool for safe multi-threading
thread_local = threading.local()

def get_db_conn():
    """Retrieves or creates a thread-local SQLite database connection."""
    if not hasattr(thread_local, "conn"):
        thread_local.conn = sqlite3.connect(
            CACHE_DB_PATH, timeout=30.0, check_same_thread=False
        )
    return thread_local.conn

def generate_cache_key(text: str, system_prompt: str) -> str:
    """Generates an MD5 hash key combining the system prompt and text input."""
    combined = f"{system_prompt}::{text}"
    return hashlib.md5(combined.encode("utf-8")).hexdigest()

def get_cached_val(text: str, system_prompt: str):
    """Retrieves cached prediction value from SQLite database if present."""
    key = generate_cache_key(text, system_prompt)
    conn = get_db_conn()
    cursor = conn.cursor()
    cursor.execute("SELECT val FROM predictions WHERE key=?", (key,))
    row = cursor.fetchone()
    return row[0] if row else None

def set_cached_val(text: str, system_prompt: str, val):
    """Stores a prediction value into the SQLite database cache."""
    key = generate_cache_key(text, system_prompt)
    conn = get_db_conn()
    with conn:
        conn.execute("INSERT OR REPLACE INTO predictions VALUES (?, ?)", (key, str(val)))

print("SQLite Prediction Cache module initialized successfully!")

SQLite Prediction Cache module initialized successfully!


### 2. Gemini API Configuration
We define the classification rules here. The prompt is designed to handle edge cases where Qatar is mentioned incidentally versus playing an active role.

In [ ]:
GEMINI_API_KEY = userdata.get("qpip")
if not GEMINI_API_KEY:
    raise ValueError("Please add GEMINI_API_KEY to your Colab Secrets.")

client = genai_new.Client(api_key=GEMINI_API_KEY)

SYSTEM_INSTRUCTION = """
You are an expert Arabic news classification assistant.
Your task is to determine if the given article is related to Qatar.

Rules:
- Output 1 if Qatar (government, economy, sports, diplomacy, residents, or events) plays a meaningful role.
- Output 1 even if other entities are mentioned, as long as Qatar's involvement is significant.
- Output 0 ONLY if Qatar is absent, a passing location in a list, or an incidental historical mention.

Respond ONLY with a single digit: 0 or 1.
"""

In [15]:
import torch
from torch.utils.data import DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# 1. Fit Label Encoders for ALL 6 Target Columns
label_encoders = {}
class_counts = {}

for column in TARGET_COLUMNS:
    le = LabelEncoder()
    # Ensure strings to prevent numeric/string type mismatches
    train_df[column] = train_df[column].astype(str)
    le.fit(train_df[column])
    label_encoders[column] = le
    class_counts[column] = len(le.classes_)

print("Target Class Counts across all 6 heads:")
for col, count in class_counts.items():
    print(f" - {col}: {count} classes")

# 2. Train / Validation Split (85% Train, 15% Validation)
train_sub, val_sub = train_test_split(train_df, test_size=0.15, random_state=SEED)

# 3. Create PyTorch Datasets & DataLoaders
train_dataset = NewsDataset(train_sub, is_test=False)
val_dataset = NewsDataset(val_sub, is_test=False)
test_dataset = NewsDataset(test_df, is_test=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# 4. Instantiate Model & Optimizer on GPU
model = MultiHeadAraBERT(class_counts=class_counts).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

print(f"\nModel successfully defined and loaded on: {device}")
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}")

Target Class Counts across all 6 heads:
 - Qatar Related: 2 classes
 - Geography: 13 classes
 - Politics & Conflict: 3 classes
 - Health & Wellbeing: 3 classes
 - Science: 4 classes
 - Sports: 5 classes


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  654MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  654MB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] BertModel LOAD REPORT from: UBC-NLP/MARBERTv2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Model successfully defined and loaded on: cuda
Train batches: 277 | Val batches: 49 | Test batches: 79


### 3. Classification Logic and Execution
We use a `ThreadPoolExecutor` to speed up processing of the dataset. Results are stored in the validation and test DataFrames.

In [16]:
import torch
import numpy as np
from tqdm.auto import tqdm
from sklearn.metrics import f1_score

# 1. Setup Model Checkpoint Path & Metric Tracking
best_mean_f1 = 0.0
best_model_path = "/content/best_marbert_model.pt"

print(f"Starting Multi-Head MARBERT Training on GPU ({device})...\n")

# 2. Training Epoch Loop
for epoch in range(EPOCHS):
    # --- TRAINING PHASE ---
    model.train()
    total_train_loss = 0.0

    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
    for batch in train_bar:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = {k: v.to(device) for k, v in batch["labels"].items()}

        # Forward Pass (Calculates loss across all 6 heads)
        loss, logits = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)

        # Backward Pass & Gradient Clipping
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_train_loss += loss.item()
        train_bar.set_postfix({"loss": f"{loss.item():.4f}"})

    avg_train_loss = total_train_loss / len(train_loader)

    # --- VALIDATION PHASE ---
    model.eval()
    val_preds = {col: [] for col in TARGET_COLUMNS}
    val_trues = {col: [] for col in TARGET_COLUMNS}
    total_val_loss = 0.0

    with torch.no_grad():
        val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]")
        for batch in val_bar:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = {k: v.to(device) for k, v in batch["labels"].items()}

            loss, logits = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_val_loss += loss.item()

            # Record argmax predictions for each target head
            for col in TARGET_COLUMNS:
                preds = logits[col].argmax(dim=1).cpu().numpy()
                val_preds[col].extend(preds)
                val_trues[col].extend(labels[col].cpu().numpy())

    avg_val_loss = total_val_loss / len(val_loader)

    # Calculate Weighted F1 per head
    scores = {}
    for col in TARGET_COLUMNS:
        scores[col] = f1_score(val_trues[col], val_preds[col], average="weighted")

    mean_f1 = float(np.mean(list(scores.values())))

    print(f"\n================ Epoch {epoch+1} Summary ================")
    print(f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Overall Mean F1: {mean_f1:.4f}")
    for col in TARGET_COLUMNS:
        print(f"  - {col}: F1 = {scores[col]:.4f}")

    # Save checkpoint if mean weighted F1 improves
    if mean_f1 > best_mean_f1:
        best_mean_f1 = mean_f1
        torch.save(model.state_dict(), best_model_path)
        print(f"\n Saved new best model checkpoint to '{best_model_path}' (Mean F1: {best_mean_f1:.4f})")
    print("=" * 55 + "\n")

Starting Multi-Head MARBERT Training on GPU (cuda)...



Epoch 1/4 [Train]:   0%|          | 0/277 [00:00<?, ?it/s]

Epoch 1/4 [Val]:   0%|          | 0/49 [00:00<?, ?it/s]


================ Epoch 1 Summary ================
Train Loss: 0.8129 | Val Loss: 0.5406 | Overall Mean F1: 0.8504
  - Qatar Related: F1 = 0.9374
  - Geography: F1 = 0.5997
  - Politics & Conflict: F1 = 0.8510
  - Health & Wellbeing: F1 = 0.9236
  - Science: F1 = 0.9098
  - Sports: F1 = 0.8812

 Saved new best model checkpoint to '/content/best_marbert_model.pt' (Mean F1: 0.8504)



Epoch 2/4 [Train]:   0%|          | 0/277 [00:00<?, ?it/s]

...

Epoch 3/4 [Val]:   0%|          | 0/49 [00:00<?, ?it/s]


================ Epoch 3 Summary ================
Train Loss: 0.3447 | Val Loss: 0.3005 | Overall Mean F1: 0.9269
  - Qatar Related: F1 = 0.9489
  - Geography: F1 = 0.8125
  - Politics & Conflict: F1 = 0.9322
  - Health & Wellbeing: F1 = 0.9734
  - Science: F1 = 0.9523
  - Sports: F1 = 0.9419

 Saved new best model checkpoint to '/content/best_marbert_model.pt' (Mean F1: 0.9269)



Epoch 4/4 [Train]:   0%|          | 0/277 [00:00<?, ?it/s]

Epoch 4/4 [Val]:   0%|          | 0/49 [00:00<?, ?it/s]


================ Epoch 4 Summary ================
Train Loss: 0.2524 | Val Loss: 0.2517 | Overall Mean F1: 0.9353
  - Qatar Related: F1 = 0.9527
  - Geography: F1 = 0.8412
  - Politics & Conflict: F1 = 0.9328
  - Health & Wellbeing: F1 = 0.9697
  - Science: F1 = 0.9642
  - Sports: F1 = 0.9513

 Saved new best model checkpoint to '/content/best_marbert_model.pt' (Mean F1: 0.9353)



## AraBERT for the remaining targets

This model fine-tunes **5 output heads** (Geography, Politics & Conflict, Health & Wellbeing, Science, and Sports). The 'Qatar Related' classification is handled separately by Gemini. By focusing AraBERT on these specific categories, we leverage the strengths of both models.

### 1. Model Initialization and Label Encoding
We define the base AraBERT model for our encoder and initialize `LabelEncoder` objects for the five categorical targets. Note that `Qatar Related` is excluded here as it is handled by the Gemini LLM.

In [17]:
import torch
import pandas as pd
from pathlib import Path

# 1. Load Best Checkpoint Weights into Multi-Head Model
best_model_path = "/content/best_marbert_model.pt"
print(f"Loading best model weights from '{best_model_path}'...")

# Instantiate architecture and load state dict onto GPU
model = MultiHeadAraBERT(class_counts=class_counts).to(device)
model.load_state_dict(torch.load(best_model_path, map_location=device))
model.eval()

print("Model successfully loaded and set to evaluation mode!")

# 2. Run Inference on Test Set across ALL 6 Target Dimensions
print("\nGenerating predictions on Test dataset...")
test_predictions_df = predict_arabert(model, test_df, is_test=True)

# 3. Format and Save Final Submission CSV
submission_file_path = OUTPUT_DIR / "submission.csv"
final_submission = make_submission(
    test_frame=test_df,
    predictions=test_predictions_df,
    output_path=submission_file_path
)

print("\n--- Submission Preview (First 5 Rows) ---")
display(final_submission.head())

print("\n--- Target Class Distributions in Submission ---")
for col in TARGET_COLUMNS:
    print(f"\n[Predicted: {col}]")
    display(final_submission[col].value_counts().to_frame("count"))

Loading best model weights from '/content/best_marbert_model.pt'...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: UBC-NLP/MARBERTv2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model successfully loaded and set to evaluation mode!

Generating predictions on Test dataset...
Submission successfully saved to: /content/submissions/submission.csv
Submission Shape: (1262, 7)

--- Submission Preview (First 5 Rows) ---


,ID,Qatar Related,Geography,Politics & Conflict,Health & Wellbeing,Science,Sports
0,jiRadWgPKa2i51Xl,0,G-9,NO_POLITICS,NO_HEALTH,SC-3,NO_SPORTS
1,IoKhm6KjwCkwEnAZ,0,G-4,NO_POLITICS,NO_HEALTH,SC-1,NO_SPORTS
2,ZihSiR1hnDU6w7Lz,1,G-3,NO_POLITICS,NO_HEALTH,SC-1,NO_SPORTS
3,X72YJR7IVU7CCKbh,1,G-4,NO_POLITICS,NO_HEALTH,NO_SCIENCE,SP-4
4,TpxFAGsHEekF2MpS,0,G-11,PC-1,NO_HEALTH,NO_SCIENCE,NO_SPORTS


...

,count
Health & Wellbeing,
NO_HEALTH,1035
H-2,127
H-1,100



[Predicted: Science]


,count
Science,
NO_SCIENCE,869
SC-1,195
SC-3,107
SC-2,91



[Predicted: Sports]


,count
Sports,
NO_SPORTS,777
SP-2,132
SP-3,130
SP-4,115
SP-1,108


### 2. Dataset Preparation
In this step, we wrap our training and validation splits into the custom `NewsDataset` class. This handles tokenization, padding, and mapping labels to the correct tensor formats for the multi-head heads.

In [18]:
# Prepare PyTorch Datasets for validation and test splits using all 6 target heads
val_dataset = NewsDataset(val_sub, is_test=False)
test_dataset = NewsDataset(test_df, is_test=True)

print("Validation and Test NewsDatasets successfully prepared.")

Validation and Test NewsDatasets successfully prepared.


In [19]:
import torch

best_model_path = "/content/best_marbert_model.pt"

# Load the trained 6-head MARBERT checkpoint into memory
model = MultiHeadAraBERT(class_counts=class_counts).to(device)
model.load_state_dict(torch.load(best_model_path, map_location=device))
model.eval()

print(f"Successfully loaded best model checkpoint from: {best_model_path}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: UBC-NLP/MARBERTv2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Successfully loaded best model checkpoint from: /content/best_marbert_model.pt


### 3. Multi-Head Training
We initialize the `MultiHeadAraBERT` model with specific head counts for each target and use the Hugging Face `Trainer` to fine-tune the model. We use a custom data collator to correctly batch the multiple label dictionaries.

In [20]:
from sklearn.metrics import f1_score

# Generate validation predictions for all 6 target dimensions natively
val_predictions = predict_arabert(model, val_sub, is_test=False)

print("--- Multi-Head MARBERT Validation Performance ---")
val_f1_scores = {}
for col in TARGET_COLUMNS:
    score = f1_score(val_sub[col].astype(str), val_predictions[col], average="weighted")
    val_f1_scores[col] = score
    print(f"{col:25}: F1 = {score:.4f}")

mean_f1 = sum(val_f1_scores.values()) / len(val_f1_scores)
print("-" * 50)
print(f"{'Overall Mean F1':25}: {mean_f1:.4f}")

--- Multi-Head MARBERT Validation Performance ---
Qatar Related            : F1 = 0.9527
Geography                : F1 = 0.8412
Politics & Conflict      : F1 = 0.9328
Health & Wellbeing       : F1 = 0.9697
Science                  : F1 = 0.9642
Sports                   : F1 = 0.9513
--------------------------------------------------
Overall Mean F1          : 0.9353


### 4. Hybrid Validation Evaluation
We evaluate the model's performance on the validation split. By combining the zero-shot LLM predictions for 'Qatar Related' with the fine-tuned AraBERT predictions for the remaining 5 targets, we get a complete picture of our multi-label accuracy.

In [21]:
from sklearn.metrics import f1_score

# 1. Use the correct validation split dataframe
val_df = val_sub if 'val_sub' in globals() else validation_part

# 2. Generate MARBERT predictions across ALL 6 target heads
arabert_val_results = predict_arabert(model, val_df, is_test=False)

# 3. Calculate Weighted F1 scores for each target column
print("--- Multi-Head MARBERT Validation Performance ---")
val_scores = {}
for col in TARGET_COLUMNS:
    score = f1_score(val_df[col].astype(str), arabert_val_results[col], average="weighted")
    val_scores[col] = score
    print(f"{col:25}: F1 = {score:.4f}")

# 4. Overall Mean F1 Score
mean_f1 = sum(val_scores.values()) / len(val_scores)
print("-" * 50)
print(f"{'Overall Mean F1':25}: {mean_f1:.4f}")

--- Multi-Head MARBERT Validation Performance ---
Qatar Related            : F1 = 0.9527
Geography                : F1 = 0.8412
Politics & Conflict      : F1 = 0.9328
Health & Wellbeing       : F1 = 0.9697
Science                  : F1 = 0.9642
Sports                   : F1 = 0.9513
--------------------------------------------------
Overall Mean F1          : 0.9353


### 5. Test Set Inference
Now we apply the AraBERT model to the competition test set to classify the 5 general categories.

In [22]:
# Run batched inference across all 6 target heads on the test set
print("Generating predictions for all 6 target heads on test set...")
arabert_test_results = predict_arabert(model, test_df, is_test=True)
print("Test set inference completed successfully.")

Generating predictions for all 6 target heads on test set...
Test set inference completed successfully.


### 6. Final Submission Construction
We merge the results from both models and save the final CSV. This ensures the output format matches the ZINDI competition requirements.

In [23]:
# Construct and export the final 6-head submission file
final_sub_path = OUTPUT_DIR / "submission.csv"
submission_df = make_submission(test_df, arabert_test_results, final_sub_path)

print(f"Final submission saved to: {final_sub_path}")
display(submission_df.head(10))

Submission successfully saved to: /content/submissions/submission.csv
Submission Shape: (1262, 7)
Final submission saved to: /content/submissions/submission.csv


,ID,Qatar Related,Geography,Politics & Conflict,Health & Wellbeing,Science,Sports
0,jiRadWgPKa2i51Xl,0,G-9,NO_POLITICS,NO_HEALTH,SC-3,NO_SPORTS
1,IoKhm6KjwCkwEnAZ,0,G-4,NO_POLITICS,NO_HEALTH,SC-1,NO_SPORTS
2,ZihSiR1hnDU6w7Lz,1,G-3,NO_POLITICS,NO_HEALTH,SC-1,NO_SPORTS
3,X72YJR7IVU7CCKbh,1,G-4,NO_POLITICS,NO_HEALTH,NO_SCIENCE,SP-4
4,TpxFAGsHEekF2MpS,0,G-11,PC-1,NO_HEALTH,NO_SCIENCE,NO_SPORTS
5,5yMi2iGblsaKEf3a,1,G-8,PC-1,NO_HEALTH,NO_SCIENCE,NO_SPORTS
6,HI94q4KffiYZI489,0,G-8,PC-2,NO_HEALTH,SC-1,NO_SPORTS
7,bW5j7uNLL18myHlg,1,G-3,NO_POLITICS,NO_HEALTH,SC-1,NO_SPORTS
8,29B6FZZR67pP440u,0,G-2,NO_POLITICS,NO_HEALTH,NO_SCIENCE,SP-3
9,VJA0AnkbOnWF9xAa,0,NO_GEOGRAPHY,NO_POLITICS,H-2,SC-2,NO_SPORTS
